<a href="https://colab.research.google.com/github/olamide-ogunbanjo/IC-Credito---Olamide-Ogunbanjo/blob/main/Extrator_IF_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q playwright
!playwright install --with-deps chromium

In [ ]:
# =============================================================================
#  Automação de extração do site oficial do IF.data
#
# Alterar manualmente o Tipo de Instituição, Relatório e Período desejados;
# Trocar a URL com base no período (antes de 2000, 2000 até 2024 e 2025 em diante, olhar a URL ao selecionar o período no site);
# Recomendo rodar no Google Colab.
#
# =============================================================================

# !pip install -q playwright
# !playwright install --with-deps chromium

import os
import re
import asyncio
from playwright.async_api import async_playwright, TimeoutError as PWTimeout

URL = "https://www3.bcb.gov.br/ifdata/index.html"

TIPO_INSTITUICAO_TEXTO = "Conglomerados Prudenciais e Instituições Independentes"
TIPO_RELATORIO_TEXTO = "Carteira de crédito ativa Pessoa Jurídica - por porte do tomador"

MODO_TESTE = False   # <<< troque pra False depois de conferir que funcionou

RAW_DIR = "ifdata_csv_site"
DEBUG_DIR = "ifdata_debug"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DEBUG_DIR, exist_ok=True)

periodos = []
for ano in range(2025, 2026):
    for mes in ("03", "06", "09", "12"):
        periodos.append(f"{mes}/{ano}")

if MODO_TESTE:
    periodos = periodos[:1]
    print(f"[MODO_TESTE ativo] Rodando só o período: {periodos[0]}\n")


async def obter_funcao_e_indice(page, ul_id, texto_alvo):
    """Lê os <li><a href="javascript:funcao(N)">texto</a></li> de uma <ul>
    e retorna (nome_funcao, N) do item cujo texto bate exatamente."""
    itens = await page.eval_on_selector_all(
        f"#{ul_id} li a",
        "els => els.map(el => ({texto: el.textContent.trim(), href: el.getAttribute('href')}))"
    )
    for it in itens:
        if it["texto"].strip() == texto_alvo.strip():
            m = re.search(r"javascript:(\w+)\((\d+)\)", it["href"] or "")
            if m:
                return m.group(1), int(m.group(2))
    return None, None


async def selecionar(page, ul_id, texto_alvo, debug_tag=None):
    funcao, idx = await obter_funcao_e_indice(page, ul_id, texto_alvo)
    if funcao is None:
        opcoes = await page.eval_on_selector_all(
            f"#{ul_id} li a", "els => els.map(el => el.textContent.trim())"
        )
        raise RuntimeError(
            f"Não achei a opção '{texto_alvo}' em #{ul_id}. "
            f"Opções disponíveis nesse momento: {opcoes}"
        )
    await page.evaluate(f"{funcao}({idx})")
    if MODO_TESTE and debug_tag:
        await page.wait_for_timeout(500)
        await page.screenshot(path=f"{DEBUG_DIR}/{debug_tag}.png")


async def baixar_periodo(page, periodo):
    print(f"[{periodo}] abrindo página...")
    await page.goto(URL, wait_until="networkidle", timeout=60000)
    if MODO_TESTE:
        await page.screenshot(path=f"{DEBUG_DIR}/00_pagina_inicial.png")

    print(f"[{periodo}] selecionando data-base...")
    await selecionar(page, "ulDataBase", periodo, "01_data_base")

    print(f"[{periodo}] aguardando lista de tipo de instituição carregar...")
    await page.wait_for_function(
        "document.querySelectorAll('#ulTipoInst li').length > 0", timeout=15000
    )
    print(f"[{periodo}] selecionando tipo de instituição...")
    await selecionar(page, "ulTipoInst", TIPO_INSTITUICAO_TEXTO, "02_tipo_instituicao")

    print(f"[{periodo}] aguardando lista de tipo de relatório carregar...")
    await page.wait_for_function(
        "document.querySelectorAll('#ulRelatorio li').length > 0", timeout=15000
    )
    print(f"[{periodo}] selecionando tipo de relatório...")
    await selecionar(page, "ulRelatorio", TIPO_RELATORIO_TEXTO, "03_tipo_relatorio")

    print(f"[{periodo}] aguardando a tabela carregar...")
    await page.wait_for_selector("#divExport", state="visible", timeout=30000)
    if MODO_TESTE:
        await page.screenshot(path=f"{DEBUG_DIR}/04_tabela_carregada.png")

    print(f"[{periodo}] baixando CSV...")
    async with page.expect_download(timeout=30000) as download_info:
        await page.evaluate("downloadCsv()")
    download = await download_info.value

    nome_arquivo = periodo.replace("/", "_") + ".csv"
    caminho_final = os.path.join(RAW_DIR, nome_arquivo)
    await download.save_as(caminho_final)
    print(f"[{periodo}] -> salvo em {caminho_final}")
    return caminho_final


async def main():
    falhas = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(accept_downloads=True)
        page = await context.new_page()

        for periodo in periodos:
            try:
                await baixar_periodo(page, periodo)
            except (PWTimeout, RuntimeError) as e:
                print(f"[{periodo}] -> FALHOU: {e}")
                try:
                    await page.screenshot(
                        path=f"{DEBUG_DIR}/ERRO_{periodo.replace('/', '_')}.png"
                    )
                except Exception:
                    pass
                falhas.append(periodo)
            await asyncio.sleep(1)

        await browser.close()

    print("\n" + "=" * 70)
    if MODO_TESTE:
        print("TESTE concluído. Confira:")
        print(f"  - O CSV baixado em: {RAW_DIR}/")
        print(f"  - Os screenshots de cada passo em: {DEBUG_DIR}/")
        print("Se deu tudo certo, troque MODO_TESTE = False e rode de novo")
        print("para baixar os 44 períodos completos.")
    else:
        print(f"Concluído. {len(periodos) - len(falhas)}/{len(periodos)} períodos baixados com sucesso.")
        if falhas:
            print(f"Falharam (confira screenshots ERRO_*.png em {DEBUG_DIR}/): {falhas}")
        print(f"\nOs CSVs individuais estão em: {RAW_DIR}/ (um por trimestre)")
        print("Depois é só juntar com pandas, por exemplo:")
        print("""
import pandas as pd, glob
arquivos = glob.glob('ifdata_csv_site/*.csv')
dfs = [pd.read_csv(f, sep=';', encoding='latin-1') for f in arquivos]
df_final = pd.concat(dfs, ignore_index=True)
df_final.to_csv('ifdata_consolidado.csv', index=False, encoding='utf-8-sig')
""")


await main()

In [ ]:
# Assimilador dos CSVs extraídos do IF Data

import pandas as pd
import csv
import glob
import os
import re

PASTA = 'ifdata_csv_site'

# colunas de identificação que TODA linha de dado real do IF.data preenche
# (usadas para detectar automaticamente se há 1 ou 2 linhas de cabeçalho,
# e para saber quais colunas nunca devem virar número)
COLUNAS_ID = ['Instituição', 'Código', 'TCB', 'TD', 'TC', 'SR', 'Segmento', 'Cidade', 'UF', 'Data']


def eh_linha_de_dado(campos, n_checar=5):
    """Uma linha de dado real preenche quase todas as primeiras colunas de
    identificação (Instituição, Código, TCB, TD, TC...). Uma linha de
    subcabeçalho (categoria mesclada) deixa essas colunas quase todas vazias."""
    primeiros = campos[:n_checar]
    preenchidos = sum(1 for v in primeiros if v.strip() != '')
    return preenchidos >= n_checar - 1  # tolera 1 célula vazia (ex: SR)


def carregar_arquivo(caminho):
    """Lê um CSV do IF.data de forma resistente a variações entre relatórios:
    - conserta o encoding (UTF-8 com BOM, não latin-1)
    - detecta sozinho se o cabeçalho tem 1 linha (relatórios simples) ou
      2 linhas (categoria + subcategoria, em relatórios com colunas agrupadas)
    - encontra e descarta o bloco de resumo (percentuais por Tipo de
      Controle/Consolidado/Consolidação) colado no final do arquivo
    """
    with open(caminho, encoding='utf-8-sig') as f:
        linhas = [l for l in f.readlines() if l.strip() != '']

    parsed = list(csv.reader(linhas, delimiter=';'))

    # --- descobre quantas linhas de cabeçalho existem ---
    header_rows = 1
    while header_rows < len(parsed) and not eh_linha_de_dado(parsed[header_rows]):
        header_rows += 1
    if header_rows >= len(parsed):
        raise ValueError(f"Não encontrei nenhuma linha de dado em {caminho}")

    # --- descobre onde termina a tabela principal (bloco de resumo começa
    # numa linha só com 1 campo, ex: "TC - Tipo de Controle") ---
    fim = len(parsed)
    for i in range(header_rows, len(parsed)):
        if len(parsed[i]) <= 1:
            fim = i
            break

    dados = parsed[header_rows:fim]
    if not dados:
        raise ValueError(f"Nenhuma linha de dados encontrada em {caminho}")
    n_cols = len(dados[0])

    # as linhas de cabeçalho costumam ter 1 campo a mais (';' sobrando no
    # final da linha) -> corta para o mesmo tamanho das linhas de dado
    niveis = [linha[:n_cols] for linha in parsed[:header_rows]]

    # preenche para a frente cada nível de cabeçalho (categorias mescladas
    # como "Veículos" que só aparecem na primeira sub-coluna do grupo)
    niveis_preenchidos = []
    for nivel in niveis:
        atual = ''
        preenchido = []
        for v in nivel:
            v = v.strip()
            if v != '':
                atual = v
            preenchido.append(atual)
        niveis_preenchidos.append(preenchido)

    colunas = []
    for i in range(n_cols):
        partes = []
        anterior = None
        for nivel in niveis_preenchidos:
            val = nivel[i]
            if val != '' and val != anterior:
                partes.append(val)
            anterior = val
        colunas.append(' - '.join(partes) if partes else f'coluna_{i}')

    # garante nomes de coluna únicos (evita erro se duas colunas derem o
    # mesmo nome combinado, ex: relatório com estrutura inesperada)
    vistos = {}
    colunas_unicas = []
    for c in colunas:
        vistos[c] = vistos.get(c, 0) + 1
        colunas_unicas.append(c if vistos[c] == 1 else f"{c} ({vistos[c]})")

    df = pd.DataFrame(dados, columns=colunas_unicas)

    nome = os.path.basename(caminho).replace('.csv', '')
    m = re.match(r'(\d{2})_(\d{4})', nome)
    if not m:
        raise ValueError(f"Nome de arquivo fora do padrão MM_AAAA.csv: {caminho}")
    mes, ano = m.group(1), m.group(2)
    df.insert(0, 'periodo', f"{mes}/{ano}")
    df.insert(1, 'ano', int(ano))
    df.insert(2, 'mes', int(mes))
    return df


def converter_numericos(df):
    """Converte colunas de valor (formato brasileiro '1.234.567' ou '12,34%')
    para número. Mantém como texto as colunas de identificação/categoria."""
    colunas_texto = {'periodo', 'ano', 'mes'} | set(COLUNAS_ID)
    for col in df.columns:
        if col in colunas_texto:
            continue
        serie = df[col].astype(str).str.strip()
        serie = serie.replace({'NA': None, 'NI': None, '': None})
        eh_percentual = serie.str.contains('%', na=False)
        serie = serie.str.replace('%', '', regex=False)
        serie = serie.str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        numerico = pd.to_numeric(serie, errors='coerce')
        if eh_percentual.any():
            numerico = numerico / 100
        df[col] = numerico
    return df


def chave_data(caminho):
    nome = os.path.basename(caminho)
    m = re.match(r'(\d{2})_(\d{4})\.csv', nome)
    if not m:
        return (9999, 99)
    mes, ano = int(m.group(1)), int(m.group(2))
    return (ano, mes)


# --- carrega todos os arquivos da pasta, em ordem cronológica ---
arquivos = sorted(glob.glob(os.path.join(PASTA, '*.csv')), key=chave_data)

print(f"Encontrados {len(arquivos)} arquivos.")
dfs = []
falhas = []
for arquivo in arquivos:
    try:
        df = carregar_arquivo(arquivo)
        print(f" - {os.path.basename(arquivo)}: OK ({df.shape[0]} linhas, {df.shape[1]} colunas)")
        dfs.append(df)
    except Exception as e:
        print(f" - {os.path.basename(arquivo)}: FALHOU -> {e}")
        falhas.append(arquivo)

if not dfs:
    raise RuntimeError("Nenhum arquivo foi carregado com sucesso.")

# concatena alinhando pelas colunas por NOME (não por posição) -- essencial
# porque relatórios diferentes / períodos diferentes podem ter colunas
# diferentes (ex: "Segmento" existe só em alguns períodos)
df_final = pd.concat(dfs, ignore_index=True, sort=False)
df_final = converter_numericos(df_final)

print(f"\nConsolidado: {df_final.shape[0]} linhas x {df_final.shape[1]} colunas")
if falhas:
    print(f"Arquivos que falharam ({len(falhas)}): {[os.path.basename(f) for f in falhas]}")

df_final.to_csv(f'{PASTA}_consolidado.csv', index=False, sep=';', encoding='utf-8-sig')
print(f"Salvo em: {PASTA}_consolidado.csv")